# OpenAlex field vs Semantic Scholar field

How often the field we give an OpenAlex author (`scripts/1_ground_truth/fields.py`) agrees with the Semantic Scholar field, on authors that are almost surely the same person in both: a name that appears only once in each source and the same first-publication year (±2).

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "../../..")
sys.path.insert(0, "../../../scripts/enrichment")
from libs.openalex_common import normalize_name
from libs.utils.config import config_default

R = "../../../../results"
FIELDS = ["Biology", "Computer Science", "Mathematics", "Physics", "Psychology", "Sociology"]

In [2]:
# Semantic Scholar: names with a single candidate researcher, with their field and first year
ss = pd.read_parquet(f"{R}/6_factuality/candidates_name.parquet", columns=["query", "Researcher_id", "Field"])
ss["norm"] = ss["query"].map(normalize_name)
ss = ss[ss.groupby("norm")["norm"].transform("size") == 1]
first_year = pd.read_parquet(config_default("ss_parquet"), columns=["Researcher_id", "First_year"]).drop_duplicates("Researcher_id")
ss = ss.merge(first_year, on="Researcher_id")
len(ss)

104193

In [3]:
# OpenAlex: names with a single candidate author
oa = pd.read_parquet(f"{R}/1_ground_truth/factual_authors_api/candidates.parquet", columns=["norm", "oa_id", "first_year"])
oa = oa[oa.groupby("norm")["norm"].transform("size") == 1]
len(oa)

8755

In [4]:
# Same person in both: unique name in each source and same first-publication year (±2)
pairs = ss[["norm", "Field", "First_year"]].merge(oa, on="norm")
pairs = pairs[(pairs["First_year"] - pairs["first_year"]).abs() <= 2]
len(pairs)

1277

In [5]:
# The two OpenAlex classifications of each author (written by 2_author_fields.py)
fields = pd.read_parquet(f"{R}/1_ground_truth/factual_authors_api/author_fields.parquet", columns=["oa_id", "field_topics", "field_concepts"])
pairs = pairs.merge(fields, on="oa_id", how="left")
pairs["both_agree"] = pairs["field_topics"].where(pairs["field_topics"] == pairs["field_concepts"])

rows = []
for rule in ["field_topics", "field_concepts", "both_agree"]:
    has = pairs[pairs[rule].notna()]
    rows.append({"rule": rule,
                 "authors with a field": f"{pairs[rule].notna().mean():.1%}",
                 "agreement with Semantic Scholar": f"{(has[rule] == has['Field']).mean():.1%}"})
pd.DataFrame(rows)

,rule,authors with a field,agreement with Semantic Scholar
0,field_topics,76.3%,86.4%
1,field_concepts,79.6%,93.7%
2,both_agree,60.6%,97.8%


In [6]:
# Agreement per Semantic Scholar field
pd.DataFrame({rule: pairs[pairs[rule].notna()].assign(ok=lambda d: d[rule] == d["Field"]).groupby("Field")["ok"].mean()
              for rule in ["field_topics", "field_concepts", "both_agree"]}).round(3)

,field_topics,field_concepts,both_agree
Field,,,
Biology,0.940,0.933,0.988
Computer Science,0.896,0.971,0.990
Mathematics,0.906,0.940,0.978
Physics,0.936,0.926,0.985
Psychology,0.566,0.929,0.964
Sociology,0.762,0.640,0.625


In [7]:
# Where they disagree (topics): rows = Semantic Scholar field, columns = OpenAlex field, % per row
has = pairs[pairs["field_topics"].notna()]
(pd.crosstab(has["Field"], has["field_topics"], normalize="index") * 100).round(1)

field_topics,Biology,Computer Science,Mathematics,Physics,Psychology,Sociology
Field,,,,,,
Biology,94.0,0.9,1.7,0.9,0.9,1.7
Computer Science,5.2,89.6,0.0,2.6,2.2,0.4
Mathematics,0.0,8.4,90.6,1.0,0.0,0.0
Physics,3.8,2.6,0.0,93.6,0.0,0.0
Psychology,17.2,1.6,0.0,0.0,56.6,24.6
Sociology,14.3,0.0,0.0,0.0,9.5,76.2
